# Gabarito: Exercício: frete grátis vale a pena?

**Disciplina 4 — Decisão com Dados · Intervalos de Confiança**

## O caso

Marketing da Olist quer lançar a promoção **"frete grátis em compras a partir de R$ 99"**. Antes de aprovar, Financeiro quer saber:

1. Quanto custa, em média, cada frete que a empresa passaria a pagar?
2. Quantos pedidos ficariam elegíveis para a promoção?
3. Quanto isso custa a cada 1.000 pedidos, em cada região?
4. Em que região a promoção cabe no orçamento?

**Orçamento aprovado por Financeiro:** até **R$ 15.000 a cada 1.000 pedidos** (R$ 15 por pedido, em média).

**Regra de decisão:** como na aula, vamos decidir pelo **cenário pessimista**, usando os limites dos ICs, e não só pela estimativa.

## Como fazer

- Cada tarefa traz uma **dica** de qual função usar. Os ICs devem ser calculados com as bibliotecas (`scipy.stats` e `statsmodels`).
- Pode escrever o código direto ou montar um prompt para a IA. Se usar IA, **leia o código** antes de rodar e confira se ele faz o que a tarefa pede.
- Depois de cada resultado, escreva **uma ou duas frases** de leitura: o que o número diz para o negócio.
- Base: pedidos **entregues**, comprados entre **janeiro e agosto de 2018** (o mesmo recorte da aula).

## 0. Carregando os dados (pronto, só rodar)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

url = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/"

pedidos = pd.read_csv(url + "olist_orders_dataset.csv")
itens = pd.read_csv(url + "olist_order_items_dataset.csv")
clientes = pd.read_csv(url + "olist_customers_dataset.csv")

pedidos["order_purchase_timestamp"] = pd.to_datetime(pedidos["order_purchase_timestamp"])

# região de cada UF
uf_para_regiao = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste",
    "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul",
}

print("pedidos:", pedidos.shape, "| itens:", itens.shape, "| clientes:", clientes.shape)

pedidos: (99441, 8) | itens: (112650, 7) | clientes: (99441, 5)


## Tarefa 1. Montar a base: um pedido por linha

A tabela `itens` tem **um item por linha**: um pedido com 3 produtos aparece 3 vezes. Para a promoção, o que importa é o **pedido inteiro**.

1. Filtre `pedidos`: `order_status == "delivered"` e compra entre `"2018-01-01"` (inclusive) e `"2018-09-01"` (exclusive).
2. Em `itens`, agrupe por `order_id` e calcule `valor_produtos` (soma de `price`) e `frete` (soma de `freight_value`).
3. Junte as três tabelas numa tabela `base`: pedidos filtrados + valores por pedido (pelo `order_id`) + `customer_state` dos clientes (pelo `customer_id`).
4. Crie as colunas:
   - `regiao`: use o dicionário `uf_para_regiao`
   - `elegivel`: True se `valor_produtos >= 99`
5. Mostre quantos pedidos ficaram e o `describe` de `valor_produtos` e `frete`.

**Dica:** `groupby("order_id").agg(...)`, `merge(..., on=...)`, `.map(uf_para_regiao)`.

**Para conferir:** devem ficar **52.783 pedidos**.

In [2]:
# 1. pedidos entregues de jan a ago de 2018
filtro = (pedidos["order_status"] == "delivered") & \
         (pedidos["order_purchase_timestamp"] >= "2018-01-01") & \
         (pedidos["order_purchase_timestamp"] < "2018-09-01")
pedidos_2018 = pedidos[filtro]

# 2. valor dos produtos e frete de cada pedido (soma dos itens)
valores = itens.groupby("order_id").agg(valor_produtos=("price", "sum"),
                                        frete=("freight_value", "sum")).reset_index()

# 3. juntando tudo
base = pedidos_2018.merge(valores, on="order_id")
base = base.merge(clientes[["customer_id", "customer_state"]], on="customer_id")

# 4. novas colunas
base["regiao"] = base["customer_state"].map(uf_para_regiao)
base["elegivel"] = base["valor_produtos"] >= 99

# 5. conferência
print("Pedidos na base:", len(base))
base[["valor_produtos", "frete"]].describe().round(2)

Pedidos na base: 52783


,valor_produtos,frete
count,52783.00,52783.00
mean,136.75,23.42
std,202.43,23.27
min,0.85,0.00
25%,45.90,13.69
50%,88.00,18.13
75%,149.90,24.26
max,7160.00,1794.96


**Leitura:** 52.783 pedidos. O pedido mediano tem R$ 88 em produtos e R$ 18 de frete. A mediana dos produtos fica **abaixo** de R$ 99, então algo perto de metade dos pedidos deve ficar elegível. O frete tem cauda longa (máximo de quase R$ 1.800), o que já avisa: a média do frete vai ficar acima da mediana.

## Tarefa 2. Quanto custa um frete? (IC para média)

1. Calcule o **frete médio por pedido** no Brasil, com IC de 95%.
2. Faça o mesmo **por região**: uma tabela com `regiao`, `n`, `frete_medio`, `li` e `ls`, ordenada pelo frete médio.

**Dica:** `stats.t.interval(0.95, df=n - 1, loc=media, scale=stats.sem(dados))`. Use um loop pelas regiões e guarde os resultados numa lista de dicionários.

**Pergunta:** em que região o frete é mais caro? A diferença para o Sudeste é grande?

In [3]:
# 1. Brasil
dados = base["frete"]
n = len(dados)
media = dados.mean()
li, ls = stats.t.interval(0.95, df=n - 1, loc=media, scale=stats.sem(dados))
print(f"Brasil: frete médio R$ {media:.2f}  IC 95% [{li:.2f} ; {ls:.2f}]")

# 2. por região
linhas = []
for regiao in base["regiao"].unique():
    dados = base[base["regiao"] == regiao]["frete"]
    n = len(dados)
    media = dados.mean()
    li, ls = stats.t.interval(0.95, df=n - 1, loc=media, scale=stats.sem(dados))
    linhas.append({"regiao": regiao, "n": n, "frete_medio": media, "li": li, "ls": ls})

frete_regiao = pd.DataFrame(linhas).sort_values("frete_medio")
frete_regiao.round(2)

Brasil: frete médio R$ 23.42  IC 95% [23.22 ; 23.62]


,regiao,n,frete_medio,li,ls
2,Sudeste,36803,20.22,20.04,20.41
3,Sul,7332,25.83,25.33,26.34
1,Centro-Oeste,3113,27.37,26.49,28.26
0,Nordeste,4669,38.24,37.03,39.46
4,Norte,866,44.49,42.22,46.76


**Leitura:** o frete médio no Brasil é de ~R$ 23,40. Mas varia muito por região: ~R$ 20 no Sudeste contra ~R$ 38 no Nordeste e ~R$ 44 no Norte, **mais que o dobro**. Os ICs não se sobrepõem. Uma promoção nacional teria custo muito diferente em cada região.

## Tarefa 3. Quantos pedidos ficam elegíveis? (IC para proporção)

1. Calcule a **proporção de pedidos elegíveis** (valor ≥ R$ 99) no Brasil, com IC de Wilson.
2. Faça o mesmo por região: `regiao`, `n`, `x` (elegíveis), `prop_%`, `li_%`, `ls_%`.

**Dica:** `proportion_confint(x, n, alpha=0.05, method="wilson")`, onde `x` é o número de elegíveis.

**Pergunta:** a proporção de elegíveis muda muito entre regiões?

In [4]:
# 1. Brasil
x = int(base["elegivel"].sum())
n = len(base)
li, ls = proportion_confint(x, n, alpha=0.05, method="wilson")
print(f"Brasil: {x / n * 100:.1f}% elegíveis  IC 95% [{li * 100:.1f}% ; {ls * 100:.1f}%]")

# 2. por região
linhas = []
for regiao in base["regiao"].unique():
    dados = base[base["regiao"] == regiao]["elegivel"]
    x = int(dados.sum())
    n = len(dados)
    li, ls = proportion_confint(x, n, alpha=0.05, method="wilson")
    linhas.append({"regiao": regiao, "n": n, "x": x,
                   "prop_%": round(x / n * 100, 1),
                   "li_%": round(li * 100, 1),
                   "ls_%": round(ls * 100, 1)})

elegiveis_regiao = pd.DataFrame(linhas).sort_values("prop_%")
elegiveis_regiao

Brasil: 45.3% elegíveis  IC 95% [44.9% ; 45.7%]


,regiao,n,x,prop_%,li_%,ls_%
2,Sudeste,36803,16060,43.6,43.1,44.1
3,Sul,7332,3432,46.8,45.7,48.0
1,Centro-Oeste,3113,1497,48.1,46.3,49.8
0,Nordeste,4669,2448,52.4,51.0,53.9
4,Norte,866,480,55.4,52.1,58.7


**Leitura:** ~45% dos pedidos ficariam elegíveis. A proporção varia pouco entre regiões (de ~44% no Sudeste a ~55% no Norte), bem menos que o frete. Quem mora longe compra pedidos um pouco maiores, talvez justamente para "compensar" o frete.

## Tarefa 4. Quanto custa a cada 1.000 pedidos? E cabe no orçamento?

A empresa só paga o frete dos pedidos **elegíveis**. Então, a cada 1.000 pedidos:

**custo = 1.000 × (proporção de elegíveis) × (frete médio dos elegíveis)**

Atenção: o frete que entra na conta é o **frete médio dos pedidos elegíveis**, não o de todos os pedidos.

Para cada região:
1. Calcule a proporção de elegíveis e seu IC de Wilson.
2. Calcule o frete médio **só dos elegíveis** e seu IC (t).
3. Calcule:
   - `custo_estimado` = 1.000 × proporção × frete médio dos elegíveis
   - `custo_pessimista` = 1.000 × **limite superior** da proporção × **limite superior** do frete
4. Crie a coluna `aprova`: True se o custo pessimista for ≤ R$ 15.000.

**Pergunta:** em quais regiões a promoção cabe no orçamento, mesmo no pior cenário?

In [5]:
orcamento = 15000   # R$ a cada 1.000 pedidos

linhas = []
for regiao in base["regiao"].unique():
    dados = base[base["regiao"] == regiao]

    # proporção de elegíveis e IC
    x = int(dados["elegivel"].sum())
    n = len(dados)
    prop = x / n
    prop_li, prop_ls = proportion_confint(x, n, alpha=0.05, method="wilson")

    # frete médio só dos elegíveis e IC
    frete_eleg = dados[dados["elegivel"] == True]["frete"]
    frete_media = frete_eleg.mean()
    frete_li, frete_ls = stats.t.interval(0.95, df=len(frete_eleg) - 1,
                                          loc=frete_media, scale=stats.sem(frete_eleg))

    # custo a cada 1.000 pedidos
    custo_estimado = 1000 * prop * frete_media
    custo_pessimista = 1000 * prop_ls * frete_ls

    linhas.append({"regiao": regiao,
                   "frete_medio_elegiveis": round(frete_media, 2),
                   "custo_estimado": round(custo_estimado),
                   "custo_pessimista": round(custo_pessimista),
                   "aprova": custo_pessimista <= orcamento})

custos = pd.DataFrame(linhas).sort_values("custo_estimado")
custos

,regiao,frete_medio_elegiveis,custo_estimado,custo_pessimista,aprova
2,Sudeste,26.88,11728,12029,True
3,Sul,32.88,15391,16232,False
1,Centro-Oeste,35.09,16873,18330,False
0,Nordeste,47.35,24825,26680,False
4,Norte,55.13,30559,34532,False


**Leitura:**

- Só o **Sudeste** cabe no orçamento, com folga: ~R$ 11.700 estimado e ~R$ 12.000 no pior cenário.
- O **Sul** fica no limite: a estimativa (~R$ 15.400) já passa dos R$ 15.000.
- Centro-Oeste, Nordeste e Norte passam bem do orçamento. No Norte, a promoção custaria o dobro.
- O frete médio dos elegíveis (~R$ 31 no Brasil) é **maior** que o de todos os pedidos (~R$ 23): pedidos maiores são mais pesados e pagam mais frete. Usar o frete médio geral subestimaria o custo.

## Tarefa 5. A diferença entre regiões é real? (IC da diferença de médias)

Marketing argumenta: "a diferença de frete entre Nordeste e Sudeste pode ser ruído". Responda com dados.

1. Calcule a diferença de frete médio **Nordeste − Sudeste**, com IC de 95%.
2. O IC contém zero?

**Dica:**

```python
from statsmodels.stats.weightstats import DescrStatsW, CompareMeans
comparacao = CompareMeans(DescrStatsW(nordeste), DescrStatsW(sudeste))
li, ls = comparacao.tconfint_diff(alpha=0.05, usevar="unequal")
```

`usevar="unequal"` não supõe que as duas regiões tenham a mesma variância.

In [6]:
from statsmodels.stats.weightstats import DescrStatsW, CompareMeans

nordeste = base[base["regiao"] == "Nordeste"]["frete"]
sudeste = base[base["regiao"] == "Sudeste"]["frete"]

diferenca = nordeste.mean() - sudeste.mean()

comparacao = CompareMeans(DescrStatsW(nordeste), DescrStatsW(sudeste))
li, ls = comparacao.tconfint_diff(alpha=0.05, usevar="unequal")

print(f"Nordeste - Sudeste: R$ {diferenca:.2f} por pedido  IC 95% [{li:.2f} ; {ls:.2f}]")

Nordeste - Sudeste: R$ 18.02 por pedido  IC 95% [16.80 ; 19.25]


**Leitura:** o frete no Nordeste custa ~R$ 18 a mais por pedido, e o IC fica longe de zero. Não é ruído: a mesma promoção custa muito mais caro no Nordeste.

## Tarefa 6. Onde o frete mais dói? (Wald × Wilson)

Um argumento a favor da promoção: em alguns pedidos, **o frete é mais caro que o próprio produto**, e isso afasta o cliente.

1. Crie a coluna `frete_caro`: True se `frete > valor_produtos`.
2. Calcule a proporção de pedidos com `frete_caro` no Brasil, com IC de Wilson.
3. Para as **5 UFs com menos pedidos**, calcule a proporção com dois ICs: Wald (`method="normal"`) e Wilson (`method="wilson"`).

**Pergunta:** o que acontece com o IC de Wald em Roraima e no Amapá? Qual dos dois você usaria para reportar à diretoria?

In [7]:
# 1. frete mais caro que o produto
base["frete_caro"] = base["frete"] > base["valor_produtos"]

# 2. Brasil
x = int(base["frete_caro"].sum())
n = len(base)
li, ls = proportion_confint(x, n, alpha=0.05, method="wilson")
print(f"Brasil: {x / n * 100:.1f}% dos pedidos com frete mais caro que o produto  IC [{li * 100:.1f}% ; {ls * 100:.1f}%]")

# 3. as 5 UFs com menos pedidos
menores = base["customer_state"].value_counts().sort_values().head(5).index

linhas = []
for uf in menores:
    dados = base[base["customer_state"] == uf]["frete_caro"]
    x = int(dados.sum())
    n = len(dados)
    wald_li, wald_ls = proportion_confint(x, n, alpha=0.05, method="normal")
    wilson_li, wilson_ls = proportion_confint(x, n, alpha=0.05, method="wilson")
    linhas.append({"uf": uf, "n": n, "x": x, "prop_%": round(x / n * 100, 1),
                   "wald": f"[{wald_li * 100:.1f}% ; {wald_ls * 100:.1f}%]",
                   "wilson": f"[{wilson_li * 100:.1f}% ; {wilson_ls * 100:.1f}%]"})

pd.DataFrame(linhas)

Brasil: 3.3% dos pedidos com frete mais caro que o produto  IC [3.1% ; 3.4%]


,uf,n,x,prop_%,wald,wilson
0,RR,22,2,9.1,[0.0% ; 21.1%],[2.5% ; 27.8%]
1,AC,27,4,14.8,[1.4% ; 28.2%],[5.9% ; 32.5%]
2,AP,39,3,7.7,[0.0% ; 16.1%],[2.7% ; 20.3%]
3,AM,72,5,6.9,[1.1% ; 12.8%],[3.0% ; 15.2%]
4,RO,109,17,15.6,[8.8% ; 22.4%],[10.0% ; 23.6%]


**Leitura:**

- No Brasil, ~3% dos pedidos têm frete mais caro que o produto. Em SP é ~1%; nos estados do Norte, a proporção é bem maior.
- Em **Roraima** (2 em 22) e no **Amapá** (3 em 39), o Wald encosta em **0%**: a conta daria negativo, e o statsmodels corta em zero. O Wilson fica acima de zero e abre mais o limite superior, como deve ser com tão poucos casos.
- Para a diretoria, reportaríamos o **Wilson**. E, com 22 pedidos, a mensagem honesta é: "em Roraima a proporção pode estar entre ~3% e ~28%; precisamos de mais dados".

## Tarefa 7. Quantos pedidos para um piloto? (tamanho de amostra)

Financeiro topa um **piloto** da promoção no Sudeste e quer estimar, durante o piloto:

- o **frete médio dos pedidos elegíveis**, com margem de **±R$ 1**;
- a **proporção de elegíveis**, com margem de **±2 pontos percentuais**.

1. Para o frete, use o desvio-padrão do frete dos elegíveis do Sudeste como palpite de σ e calcule n = (z · σ / E)², com z = 1,96 e E = 1. Arredonde para cima.
2. Para a proporção, use a proporção de elegíveis do Sudeste como palpite e calcule o n com `samplesize_confint_proportion`.
3. Quantos pedidos o piloto precisa, considerando as duas metas?

**Dica:**

```python
from statsmodels.stats.proportion import samplesize_confint_proportion
n = samplesize_confint_proportion(proportion=p, half_length=0.02, alpha=0.05)
```

In [8]:
from statsmodels.stats.proportion import samplesize_confint_proportion

sudeste = base[base["regiao"] == "Sudeste"]

# 1. frete médio dos elegíveis com ±R$ 1
sigma = sudeste[sudeste["elegivel"] == True]["frete"].std()
z = 1.96
n_frete = int(np.ceil((z * sigma / 1) ** 2))
print(f"Desvio-padrão do frete dos elegíveis no Sudeste: R$ {sigma:.2f}")
print(f"n para o frete médio com ±R$ 1: {n_frete} pedidos elegíveis")

# 2. proporção de elegíveis com ±2 p.p.
p = sudeste["elegivel"].mean()
n_prop = int(np.ceil(samplesize_confint_proportion(proportion=p, half_length=0.02, alpha=0.05)))
print(f"n para a proporção de elegíveis com ±2 p.p.: {n_prop} pedidos")

Desvio-padrão do frete dos elegíveis no Sudeste: R$ 24.15
n para o frete médio com ±R$ 1: 2241 pedidos elegíveis
n para a proporção de elegíveis com ±2 p.p.: 2363 pedidos


**Leitura:**

- A proporção de elegíveis pede ~2.400 pedidos no total.
- O frete médio pede ~2.240 pedidos **elegíveis**. Como só ~44% dos pedidos do Sudeste são elegíveis, o piloto precisa de cerca de **5.100 pedidos no total** para ter esses elegíveis.
- O frete é a meta que manda: tem cauda longa (desvio-padrão alto), então exige mais dado para a mesma precisão. Se o piloto ficar caro demais, a conversa com Financeiro é afrouxar a margem de ±R$ 1 para ±R$ 2, o que divide o n por 4.

## Tarefa 8. A recomendação

Escreva, em 3 ou 4 frases, a recomendação para Financeiro e Marketing. Use os números e os ICs que você calculou. Responda:

- Lançar a promoção? Onde?
- Por que não nas outras regiões?
- Qual é o próximo passo?

**Exemplo de resposta:**

> Recomendamos lançar o frete grátis acima de R$ 99 **só no Sudeste**, onde o custo fica em ~R$ 11.700 a cada 1.000 pedidos e não passa de ~R$ 12.000 mesmo no pior cenário, dentro do orçamento de R$ 15.000. No Sul o custo já está no limite, e no Nordeste, Norte e Centro-Oeste o frete é bem mais caro (no Nordeste, R$ 18 a mais por pedido, diferença que não é ruído), o que levaria o custo acima do orçamento. O próximo passo é um piloto no Sudeste com cerca de 5.100 pedidos, para confirmar o custo e medir se a promoção aumenta as vendas, que é o que vai dizer se ela se paga. Para as outras regiões, a alternativa é um valor mínimo de compra maior, por exemplo R$ 150, que reduziria a proporção de elegíveis.

---
## Desafios (opcional)

1. **Valor mínimo por região.** Para o Nordeste, qual valor mínimo de compra (R$ 99, 150, 200, 250...) faria a promoção caber no orçamento no cenário pessimista?
2. **Limitação.** A conta supõe que o comportamento do cliente não muda com a promoção. O que provavelmente vai acontecer com a proporção de elegíveis quando a promoção existir? Isso aumenta ou diminui o custo?